# Inspect Words MVP Database

这个 notebook 用于查看当前 SQLite 数据库内容，重点检查义项级状态是否正确更新。

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

PROJECT_ROOT = Path.cwd()
DB_PATH = PROJECT_ROOT / "data" / "words_mvp_v2.sqlite3"
DB_PATH

In [ ]:
assert DB_PATH.exists(), f"Database not found: {DB_PATH}"
con = sqlite3.connect(DB_PATH)
con.row_factory = sqlite3.Row

def q(sql, params=None):
    return pd.read_sql_query(sql, con, params=params or {})

## 1. 查看当前有哪些表

In [ ]:
q("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name
""")

## 2. 查看每张核心表的数据量

In [ ]:
tables = [
    "documents",
    "lexemes",
    "word_senses",
    "text_occurrences",
    "user_sense_states",
    "user_sense_events",
]

rows = []
for table in tables:
    count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    rows.append({"table": table, "row_count": count})
pd.DataFrame(rows)

## 3. 最近导入的文档

In [ ]:
q("""
SELECT id, filename, LENGTH(content) AS content_length, created_at
FROM documents
ORDER BY id DESC
LIMIT 20
""")

## 4. 词条与词条常见程度

In [ ]:
q("""
SELECT id, lemma, pos, frequency_rank, frequency_score, frequency_source, created_at
FROM lexemes
ORDER BY frequency_rank IS NULL, frequency_rank ASC, lemma ASC
LIMIT 50
""")

## 5. 词条对应的具体义项

In [ ]:
q("""
SELECT
    ws.id AS sense_id,
    l.lemma,
    ws.meaning_zh,
    ws.definition_en,
    ws.pos,
    ws.source,
    ws.source_sense_id,
    ws.sense_rank,
    ws.sense_key
FROM word_senses ws
JOIN lexemes l ON l.id = ws.lexeme_id
ORDER BY l.lemma ASC, ws.sense_rank ASC
LIMIT 100
""")

## 6. 文本中实际出现的位置

In [ ]:
q("""
SELECT
    o.id AS occurrence_id,
    o.document_id,
    l.lemma,
    o.surface,
    o.sentence_index,
    o.sentence,
    o.created_at
FROM text_occurrences o
LEFT JOIN lexemes l ON l.id = o.lexeme_id
ORDER BY o.id DESC
LIMIT 50
""")

## 7. 用户对具体义项的当前状态

In [ ]:
q("""
SELECT
    uss.user_id,
    uss.sense_id,
    l.lemma,
    ws.meaning_zh,
    uss.status,
    uss.mastery_level,
    uss.source_document_id,
    uss.source_occurrence_id,
    uss.last_seen_at,
    uss.last_action_at,
    uss.updated_at
FROM user_sense_states uss
JOIN word_senses ws ON ws.id = uss.sense_id
JOIN lexemes l ON l.id = ws.lexeme_id
ORDER BY uss.updated_at DESC, uss.id DESC
""")

## 8. 用户操作历史

In [ ]:
q("""
SELECT
    e.id,
    e.user_id,
    e.sense_id,
    l.lemma,
    ws.meaning_zh,
    e.event_type,
    e.from_status,
    e.to_status,
    e.document_id,
    e.occurrence_id,
    e.created_at
FROM user_sense_events e
JOIN word_senses ws ON ws.id = e.sense_id
JOIN lexemes l ON l.id = ws.lexeme_id
ORDER BY e.id DESC
LIMIT 100
""")

## 9. 检查某个词的所有义项和用户状态

修改 `target_lemma` 后重新运行下面的 cell。

In [ ]:
target_lemma = "station"

q("""
SELECT
    l.lemma,
    ws.id AS sense_id,
    ws.meaning_zh,
    ws.definition_en,
    ws.sense_rank,
    COALESCE(uss.status, 'no_user_state') AS user_status,
    uss.mastery_level,
    uss.updated_at
FROM lexemes l
JOIN word_senses ws ON ws.lexeme_id = l.id
LEFT JOIN user_sense_states uss ON uss.sense_id = ws.id AND uss.user_id = 'default'
WHERE l.lemma = :lemma
ORDER BY ws.sense_rank ASC
""", {"lemma": target_lemma})

## 10. 检查某个 occurrence 的上下文

In [ ]:
target_occurrence_id = 1

q("""
SELECT
    o.id AS occurrence_id,
    l.lemma,
    o.surface,
    o.sentence,
    o.context
FROM text_occurrences o
LEFT JOIN lexemes l ON l.id = o.lexeme_id
WHERE o.id = :occurrence_id
""", {"occurrence_id": target_occurrence_id})

## 11. 关闭数据库连接

In [ ]:
con.close()